1. Install packages


In [1]:
!pip install -q datasets transformers pandas scikit-learn

2. Load EESA CSV files

In [2]:
import pandas as pd

# Load EESA pre-split datasets (no header, columns: text, label)
train_df = pd.read_csv("EESA-Train.csv", header=None, names=["text", "label"])
val_df   = pd.read_csv("EESA-Dev.csv",   header=None, names=["text", "label"])
test_df  = pd.read_csv("EESA-Test.csv",  header=None, names=["text", "label"])

print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)
train_df.head()

Train: (2464, 2)
Val:   (818, 2)
Test:  (818, 2)


,text,label
0,retarded people be like : \n1. تحية من أرض الم...,negative
1,*أنسّى الألمَ ؟ أنها مجرد حياهَ ولن تدوّم ..* ...,neutral
2,الحراميه اللي سارقه الموسيقي من اغنيه فنانه كو...,negative
3,مش عارف ليه حاس اني داخل pupge 😂😂😂,neutral
4,500 مليون ههههههه والله العظيم ماتستاهل حتى 50...,negative


3. Clean text

In [3]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"@\w+", "", text)              # remove mentions
    text = re.sub(r"\s+", " ", text).strip()      # normalize whitespace
    text = re.sub(r"[إأآ]", "ا", text)            # normalize Arabic alef
    text = re.sub(r"ى", "ي", text)                # normalize alef maqsura
    return text

train_df["text"] = train_df["text"].apply(clean_text)
val_df["text"]   = val_df["text"].apply(clean_text)
test_df["text"]  = test_df["text"].apply(clean_text)

print("Cleaning done.")
train_df.head()

Cleaning done.


,text,label
0,retarded people be like : 1. تحية من ارض المحش...,negative
1,*انسّي الالمَ ؟ انها مجرد حياهَ ولن تدوّم ..* ...,neutral
2,الحراميه اللي سارقه الموسيقي من اغنيه فنانه كو...,negative
3,مش عارف ليه حاس اني داخل pupge 😂😂😂,neutral
4,500 مليون ههههههه والله العظيم ماتستاهل حتي 50...,negative


4. Label Encoding - Map string labels to integers

In [4]:
label_map = {"negative": 0, "neutral": 1, "positive": 2}

train_df["label"] = train_df["label"].map(label_map)
val_df["label"]   = val_df["label"].map(label_map)
test_df["label"]  = test_df["label"].map(label_map)

# Verify no NaN
print("Train nulls:", train_df["label"].isna().sum())
print("Val nulls:  ", val_df["label"].isna().sum())
print("Test nulls: ", test_df["label"].isna().sum())


Train nulls: 0
Val nulls:   0
Test nulls:  0


5. Convert to HuggingFace Dataset

In [5]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False)
test_ds  = Dataset.from_pandas(test_df,  preserve_index=False)

print(train_ds)
print(val_ds)
print(test_ds)

Dataset({
    features: ['text', 'label'],
    num_rows: 2464
})
Dataset({
    features: ['text', 'label'],
    num_rows: 818
})
Dataset({
    features: ['text', 'label'],
    num_rows: 818
})


6. Load XLM-RoBERTa tokenizer

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

7. Tokenize all splits

In [7]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_tok = train_ds.map(tokenize, batched=True)
val_tok   = val_ds.map(tokenize,   batched=True)
test_tok  = test_ds.map(tokenize,  batched=True)

Map:   0%|          | 0/2464 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

8. Remove raw text column

In [8]:
train_tok = train_tok.remove_columns(["text"])
val_tok   = val_tok.remove_columns(["text"])
test_tok  = test_tok.remove_columns(["text"])

9. Set format to torch

In [9]:
train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")

10. save dataset

In [10]:
train_tok.save_to_disk("train_dataset")
val_tok.save_to_disk("val_dataset")
test_tok.save_to_disk("test_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/2464 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/818 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/818 [00:00<?, ? examples/s]

11. Save tokenizer

In [11]:
tokenizer.save_pretrained("tokenizer_xlmr")

('tokenizer_xlmr/tokenizer_config.json', 'tokenizer_xlmr/tokenizer.json')

12. Check dataset sizes

In [12]:
print("Train size:", len(train_tok))
print("Val size:  ", len(val_tok))
print("Test size: ", len(test_tok))

Train size: 2464
Val size:   818
Test size:  818


13. Check dataset structure

In [13]:
print(train_tok[0])

{'label': tensor(0), 'input_ids': tensor([     0,  91010,    297,   3395,    186,   1884,    152,    615,  31910,
           648,    230, 144509,    906, 200317,    787,  96178,  38180,  35336,
           240, 125800,   1031,  54186,    754,  22963,    368,   1533,    716,
           477,  61591,  26894,    359,   9688,    304,  28521,    230,   1327,
         19931,   6861,  26059,      2,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
             1,      1,      1,      1,      1,      1,      1,      1,      1,
      

In [14]:
!pip install nlpaug

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 22.8 MB/s eta 0:00:00


In [15]:
import pandas as pd
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.char as nac

class SyntheticAugmentor:
    def __init__(self, target_count):
        self.target_count = target_count
        # Random word swap augmenter - works for any language including Arabic
        self.augmenter = naw.RandomWordAug(action="swap")

    def augment_text(self, text):
        try:
            result = self.augmenter.augment(text)
            return result[0] if isinstance(result, list) else result
        except:
            return text

    def augment(self, df, label_col="label", text_col="text"):
        augmented_rows = []

        for label in sorted(df[label_col].unique()):
            class_df = df[df[label_col] == label]
            current_count = len(class_df)
            needed = self.target_count - current_count

            if needed <= 0:
                print(f"  Label {label}: {current_count} — no augmentation needed")
                continue

            print(f"  Label {label}: {current_count} → {self.target_count} (+{needed})")
            samples = class_df.sample(n=needed, replace=True, random_state=42)

            for _, row in samples.iterrows():
                new_text = self.augment_text(row[text_col])
                augmented_rows.append({text_col: new_text, label_col: label})

        augmented_df = pd.DataFrame(augmented_rows)
        balanced_df = pd.concat([df, augmented_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
        return balanced_df

In [16]:
target = train_df["label"].value_counts().max()

print("Augmenting training set...")
augmentor = SyntheticAugmentor(target_count=target)
train_df = augmentor.augment(train_df)

print(f"\nNew train size: {len(train_df)}")

Augmenting training set...
  Label 0: 594 → 1092 (+498)
  Label 1: 778 → 1092 (+314)
  Label 2: 1092 — no augmentation needed

New train size: 3276


In [17]:
label_names = {0: "negative", 1: "neutral", 2: "positive"}

print("Balanced Train label distribution:")
counts = train_df["label"].value_counts().sort_index()
for label_num, count in counts.items():
    print(f"  {label_names[label_num]}: {count}")

Balanced Train label distribution:
  negative: 1092
  neutral: 1092
  positive: 1092


14.  Check label distribution

In [18]:
print("Train label distribution:")
print(train_df["label"].value_counts())

print("\nVal label distribution:")
print(val_df["label"].value_counts())

print("\nTest label distribution:")
print(test_df["label"].value_counts())

Train label distribution:
label
0    1092
2    1092
1    1092
Name: count, dtype: int64

Val label distribution:
label
2    363
1    258
0    197
Name: count, dtype: int64

Test label distribution:
label
2    363
1    258
0    197
Name: count, dtype: int64


In [19]:
from google.colab import files
import shutil

shutil.make_archive("train_dataset", "zip", "/content/train_dataset")
shutil.make_archive("val_dataset", "zip", "/content/val_dataset")
shutil.make_archive("test_dataset", "zip", "/content/test_dataset")

files.download("train_dataset.zip")
files.download("val_dataset.zip")
files.download("test_dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>